# Chapter 10D — Stage D: Autonomous Layout Synthesis

**Multi-Agent Analog EDA — Pedagogical Project (PhD track)**

---

This notebook closes the pedagogical arc from **verified schematic** to **physical realization**: *autonomous layout synthesis* under **DRC** (Design Rule Check) and **LVS** (Layout vs Schematic), with **multi-agent orchestration** over open tools and constraint extractors.

### Learning objectives

1. **Formalize** the mapping $G_{\mathrm{sch}} \mapsto L$ from verified netlist graph to geometric layout subject to PDK rules $\mathcal{R}_{\mathrm{DRC}}$ and isomorphism checks $\mathcal{R}_{\mathrm{LVS}}$.
2. **Survey** **ALIGN** (Analog Layout Intelligently Generated Automatically) and **OpenROAD** for mixed-signal digital shells, and how agents **invoke** them via **MCP tools** or **CLI subprocesses**.
3. **Encode** **constraint-driven analog layout**: symmetry, **common-centroid** matching, **guard rings**, and **proximity** for mirrors—implemented as **netlist-derived constraint objects**.
4. **Build** a **DRC proofreading agent**: parse categorized violations, traverse rule families systematically, and emit **actionable fix deltas** (spacing, width, enclosure).
5. **Optimize** **analog block placement** with **simulated annealing** on a composite cost: **wirelength + symmetry penalty + area**, and **visualize** convergence.
6. **Trace** an **end-to-end** flow: *netlist → constraints → placement → routing (abstract) → DRC/LVS → sign-off* with a **pipeline diagram**.

### Notation

- Schematic graph $G=(V,E)$ with devices $V$, nets $E$.
- Layout $L=\{(x_i,y_i,\theta_i)\}$ placements and orientations (abstracted here as centroids).
- DRC report multiset $\mathcal{V}=\{v_k\}$ of violations; **proofreading** is an ordered policy $\sigma$ over partitions $\mathcal{V}=\bigcup_c \mathcal{V}_c$ by rule category $c$.
- SA state $s_t$, temperature $T_t$, acceptance $\alpha=\min(1,\exp(-\Delta/T_t))$.

> **Disclaimer:** Toolchain calls are **mocked** (no ALIGN/OpenROAD binaries required). Replace subprocess/MCP stubs with your PDK-qualified flow while preserving interfaces.

---


In [ ]:
# Imports, RNG, dark-theme defaults (matplotlib #0d1117, plotly plotly_dark)
from __future__ import annotations

import math
import re
import textwrap
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt

import plotly.graph_objects as go
import plotly.io as pio

RNG = np.random.default_rng(2026)

DARK_BG = "#0d1117"
ACCENT = "#58a6ff"
GREEN = "#3fb950"
AMBER = "#d29922"
RED = "#f85149"
PURPLE = "#bc8cff"
CYAN = "#79c0ff"

MPL_RC = {
    "figure.facecolor": DARK_BG,
    "axes.facecolor": DARK_BG,
    "axes.edgecolor": "#30363d",
    "axes.labelcolor": "#c9d1d9",
    "text.color": "#c9d1d9",
    "xtick.color": "#8b949e",
    "ytick.color": "#8b949e",
    "grid.color": "#21262d",
    "grid.alpha": 0.65,
    "legend.facecolor": "#161b22",
    "legend.edgecolor": "#30363d",
    "font.size": 11,
}
mpl.rcParams.update(MPL_RC)
pio.templates.default = "plotly_dark"

print("Stage D environment ready | matplotlib facecolor", DARK_BG, "| plotly", pio.templates.default)


## 10D.1 Problem statement — from verified netlist to sign-off layout

**Engineering goal:** translate a **functionally verified** netlist (Stage B/C) into a **manufacturable** layout $L^\star$ such that:

1. **DRC:** $L^\star \models \mathcal{R}_{\mathrm{DRC}}$ (width/spacing/enclosure/density/antenna/… as encoded by the PDK deck).
2. **LVS:** there exists a **structure-preserving correspondence** between extracted connectivity $\mathcal{E}(L^\star)$ and the golden schematic $G_{\mathrm{sch}}$ (device recognition, pin mapping, series/parallel folding semantics).

**Analog-specific imperatives** (beyond Boolean digital correctness):

| Concern | Mechanism | Layout lever |
|--------:|-----------|--------------|
| **Matching** | $\sigma(\Delta V_t)$, gradient fields | **Symmetry**, **common-centroid**, interdigitated fingers |
| **Noise / injection** | substrate/digitally coupled disturbance | **Guard rings**, deep N-well, **shielding** routing |
| **Current fidelity** | $\lambda$-dependence, $\Delta L$ effects | **Proximity** for mirrors, shared centroid, identical orientation |
| **Stress / WPE** | STI/OS proximity | **Regularity**, dummy devices, **keep-out** patterns |

**Agentic view:** treat $(G_{\mathrm{sch}}, \Phi)$—netlist plus performance spec—as **state**; actions include *constraint edits*, *placement moves*, *routing style choices*, and *verification tool calls*; observations are **DRC/LVS/RC reports** and **extracted parasitics**.

---


## 10D.2 Layout tools overview — ALIGN, OpenROAD, and agent interfaces

### ALIGN (Analog Layout Intelligently Generated Automatically)

- **Role:** technology-aware **hierarchical analog/mixed-signal layout generation** from **structural netlists** + **user constraints** (symmetry groups, matching, block bounds).
- **Agent integration patterns:**
  - **CLI / Makefile targets:** agents emit **JSON constraint files** + netlists, then call batch drivers (project-specific) capturing `stdout/stderr` and artifact hashes.
  - **MCP wrapper:** expose tools `align.compile_constraints`, `align.run_block`, `align.fetch_artifacts` returning paths + summarized logs (token-bounded for LLM context).

### OpenROAD (digital / mixed-signal platform)

- **Role:** **floorplan, placement, CTS, routing** for standard-cell digital and **mixed-signal SoC integration** (power mesh, pin access to analog macros).
- **Agent integration:** scripted **Tcl** flows (`openroad script.tcl`) or Python bindings where available; MCP tools mirror **one logical step per tool** (e.g., `openroad.global_route`) with structured metrics (WL, DRV count).

### MCP vs command-line (design pattern)

| Aspect | CLI subprocess | MCP server |
|--------|----------------|------------|
| **Pros** | universal, easy sandboxing | **typed schemas**, discovery, auth |
| **Cons** | ad-hoc parsing | requires server maintenance |
| **Agent loop** | parse logs → JSON | model calls **functions** directly |

Below we stub **both** styles without external services.

---


In [ ]:
# --- Tool interface stubs: CLI-style + MCP-shaped calls (no external binaries) ---

from dataclasses import dataclass, field
from typing import Dict, List


@dataclass
class ToolResult:
    ok: bool
    stdout: str
    stderr: str
    artifacts: Dict[str, str] = field(default_factory=dict)


def run_cli_mock(argv: List[str]) -> ToolResult:
    # Pretend subprocess: deterministic echo of intent.
    cmd = " ".join(argv)
    return ToolResult(True, f"[mock-cli] executed: {cmd}\nexit 0\n", "")


def mcp_align_run_block(block: str, constraints_path: str) -> ToolResult:
    # MCP-shaped call: name + JSON args -> same logical effect as CLI.
    return run_cli_mock(["align", "run", "--block", block, "--constraints", constraints_path])


def mcp_openroad_floorplan(def_path: str, util: float) -> ToolResult:
    return run_cli_mock(["openroad", "-no_init", def_path, "-util", f"{util:.2f}"])


tr_align = mcp_align_run_block("ota_core", "/tmp/ota_constraints.json")
tr_or = mcp_openroad_floorplan("/tmp/chip_top.def", util=0.62)
print(tr_align.stdout)
print(tr_or.stdout)


## 10D.3 Constraint-driven analog layout — theory → netlist extraction

**Symmetry (differential pair):** enforce an axis $x=x_0$ such that devices $(i,j)$ satisfy $(x_i+x_j)/2 \approx x_0$ and **mirrored** proximity to critical nets (tail, loads).

**Common-centroid:** partition $2k$ matched devices into a **centroid matrix** to cancel linear gradients; captured as **row/column swap constraints** and **interdigitation order**.

**Guard ring:** closed **protection structure** (e.g., substrate tie ring) enclosing sensitive nets; modeled as **enclosure constraints** + **spacing** to aggressors.

**Proximity (current mirror):** minimize $\lVert \mathbf{c}_i-\mathbf{c}_j\rVert$ for devices sharing **gate net** (assuming identical orientation / STI context).

We implement **`extract_constraints(netlist_text)`** that returns a **`LayoutConstraintSet`**: symmetry pairs, mirror clusters, guard targets, and optional common-centroid groupings inferred heuristically from SPICE-like lines.

---


In [ ]:
# --- Constraint extraction from a self-contained SPICE-like netlist ---

@dataclass
class MosDevice:
    name: str
    model: str
    nodes: Tuple[str, str, str, str]  # D G S B
    w_um: float
    l_um: float


@dataclass
class LayoutConstraintSet:
    symmetry_pairs: List[Tuple[str, str]]
    mirror_groups: List[List[str]]
    guard_ring_around: List[str]
    common_centroid_groups: List[List[str]]
    notes: List[str]


_NET_RE = re.compile(
    r"^\s*X(?P<name>\w+)\s+"
    r"(?P<d>\S+)\s+(?P<g>\S+)\s+(?P<s>\S+)\s+(?P<b>\S+)\s+"
    r"(?P<model>\S+)"
    r"(?:\s+[wW]=(?P<w>[\d.]+))?"
    r"(?:\s+[lL]=(?P<l>[\d.]+))?",
    re.IGNORECASE,
)


def parse_devices(netlist: str) -> Dict[str, MosDevice]:
    devs: Dict[str, MosDevice] = {}
    for line in netlist.splitlines():
        line = line.strip()
        if not line or line.startswith("*") or line.startswith("//"):
            continue
        m = _NET_RE.match(line)
        if not m:
            continue
        w = float(m.group("w") or 2.0)
        l = float(m.group("l") or 0.15)
        d = MosDevice(
            name=m.group("name").upper(),
            model=m.group("model"),
            nodes=(m.group("d"), m.group("g"), m.group("s"), m.group("b")),
            w_um=w,
            l_um=l,
        )
        devs[d.name] = d
    return devs


def extract_constraints(netlist: str) -> LayoutConstraintSet:
    d = parse_devices(netlist)
    by_gate: Dict[str, List[str]] = {}
    for name, dv in d.items():
        by_gate.setdefault(dv.nodes[1], []).append(name)

    symmetry_pairs: List[Tuple[str, str]] = []
    mirror_groups: List[List[str]] = []
    cc_groups: List[List[str]] = []
    notes: List[str] = []

    diff_tokens = [("INP", "INM"), ("IN_P", "IN_N"), ("VP", "VN")]
    for a, b in diff_tokens:
        ga = [n for n in by_gate.get(a, []) if n in d]
        gb = [n for n in by_gate.get(b, []) if n in d]
        if ga and gb:
            symmetry_pairs.append((ga[0], gb[0]))
            cc_groups.append(sorted([ga[0], gb[0]]))
            notes.append(f"symmetry: inferred diff pair on gates {a}/{b}")
        ga_n = [n for n in by_gate.get(a, []) if n in d and "nfet" in d[n].model.lower()]
        gb_n = [n for n in by_gate.get(b, []) if n in d and "nfet" in d[n].model.lower()]
        if ga_n and gb_n:
            pair_n = (ga_n[0], gb_n[0])
            rev = (pair_n[1], pair_n[0])
            if pair_n not in symmetry_pairs and rev not in symmetry_pairs:
                symmetry_pairs.append(pair_n)
                cc_groups.append(sorted([pair_n[0], pair_n[1]]))
                notes.append(f"symmetry: inferred NFET diff pair on gates {a}/{b}")

    for g, names in by_gate.items():
        uniq = sorted(set(names))
        if len(uniq) >= 2 and g.upper() not in {"INP", "INM", "IN_P", "IN_N", "VP", "VN"}:
            mirror_groups.append(uniq)
            notes.append(f"mirror cluster on net {g}: {uniq}")

    tail_candidates = {"TAIL", "IBIAS", "ISS"}
    guard: List[str] = []
    for nm, dv in d.items():
        if any(t in dv.nodes for t in tail_candidates):
            guard.append(nm)
    if guard:
        notes.append("guard_ring: enclosing devices touching tail/bias nets")

    return LayoutConstraintSet(
        symmetry_pairs=symmetry_pairs,
        mirror_groups=mirror_groups,
        guard_ring_around=sorted(set(guard)),
        common_centroid_groups=cc_groups,
        notes=notes,
    )


EXAMPLE_NETLIST = textwrap.dedent('''
* Pedagogical OTA input stage (SPICE-like)
.subckt ota_in INP INM OUTP OUTN TAIL VDD VSS
XMP1 OUTP INP NET1 VDD pfet_01v8 W=2 L=0.15
XMP2 OUTN INM NET1 VDD pfet_01v8 W=2 L=0.15
XMN1 OUTP INP TAIL VSS nfet_01v8 W=3 L=0.15
XMN2 OUTN INM TAIL VSS nfet_01v8 W=3 L=0.15
XMTL TAIL IBIAS VSS VSS nfet_01v8 W=6 L=0.30
* Cascode bias mirror (shared gate IBIAS → mirror proximity cluster)
XMB1 NBIASA IBIAS VSS VSS nfet_01v8 W=1 L=0.5
XMB2 NBIASB IBIAS VSS VSS nfet_01v8 W=1 L=0.5
.ends ota_in
''').strip()

cs = extract_constraints(EXAMPLE_NETLIST)
print("--- Parsed devices ---")
for k, v in parse_devices(EXAMPLE_NETLIST).items():
    print(k, v)
print("\n--- Extracted constraints ---")
print("symmetry_pairs:", cs.symmetry_pairs)
print("mirror_groups:", cs.mirror_groups)
print("guard_ring_around:", cs.guard_ring_around)
print("common_centroid_groups:", cs.common_centroid_groups)
for n in cs.notes:
    print("•", n)


## 10D.4 DRC/LVS agent with **proofreading** strategy

**Observation:** raw DRC decks emit **hundreds** of redundant markers. LLM agents benefit from a **structured reading policy**:

1. **Partition** violations $\mathcal{V}$ into categories $c \in \{\text{spacing}, \text{width}, \text{enclosure}, \text{area}, \text{density}, \ldots\}$.
2. **Sort** within category by **severity** (e.g., min margin) and **layer** (diffusion vs metal).
3. **For each category**, apply **specialized fix operators** $\mathcal{F}_c$ (move edge, widen, add halo, add via array).
4. **Re-run** incremental DRC on **bounding boxes** of touched tiles when available.

**LVS** follows a parallel pattern: **device count mismatch** → check folding; **net mismatch** → probe short/open in abstract router; **pin swap** → update symmetry map.

We implement a **full mock pipeline**: synthetic report → **parser** → **proofreader** → **fix suggestions**.

---


In [ ]:
# --- Mock DRC generation, parsing, categorized proofreading, fix suggestions ---

from typing import Sequence


@dataclass
class DRCViolation:
    rule_id: str
    category: str
    layer: str
    measured_um: float
    required_um: float
    shapes: Tuple[str, str]
    margin_um: float  # negative => violation depth


_RULE_PATTERNS: List[Tuple[str, str, re.Pattern]] = [
    ("SP.1", "spacing", re.compile(
        r"spacing\s+(?P<meas>[\d.]+)\s*um\s*<\s*(?P<req>[\d.]+)\s*um\s+layer\s+(?P<layer>\w+)\s+between\s+(?P<a>\w+)\s+and\s+(?P<b>\w+)",
        re.I,
    )),
    ("WD.1", "width", re.compile(
        r"width\s+(?P<meas>[\d.]+)\s*um\s*<\s*(?P<req>[\d.]+)\s*um\s+on\s+(?P<layer>\w+)\s+shape\s+(?P<a>\w+)",
        re.I,
    )),
    ("ENC.1", "enclosure", re.compile(
        r"enclosure\s+(?P<meas>[\d.]+)\s*um\s*<\s*(?P<req>[\d.]+)\s*um\s+"
        r"(?P<inner>\w+)\s+inside\s+(?P<outer>\w+)\s+between\s+(?P<a>\w+)\s+and\s+(?P<b>\w+)",
        re.I,
    )),
]


def synthesize_mock_drc_report(n: int = 12, seed: int = 11) -> str:
    rng = np.random.default_rng(seed)
    layers = ["licon", "mcon", "met1", "met2", "diff", "tap", "via1"]
    lines: List[str] = []
    for i in range(n):
        layer = layers[int(rng.integers(0, len(layers)))]
        a, b = f"SHAPE_{i}A", f"SHAPE_{i}B"
        kind = int(rng.integers(0, 3))
        if kind == 0:
            req = float(rng.choice([0.15, 0.17, 0.20]))
            meas = max(0.02, req - float(rng.uniform(0.02, 0.06)))
            lines.append(f"VIOLATION[{i}]: spacing {meas:.3f} um < {req:.3f} um layer {layer} between {a} and {b}")
        elif kind == 1:
            req = float(rng.choice([0.13, 0.14, 0.16]))
            meas = max(0.02, req - float(rng.uniform(0.02, 0.05)))
            lines.append(f"VIOLATION[{i}]: width {meas:.3f} um < {req:.3f} um on {layer} shape {a}")
        else:
            req = float(rng.choice([0.05, 0.055, 0.07]))
            meas = max(0.01, req - float(rng.uniform(0.01, 0.03)))
            lines.append(
                f"VIOLATION[{i}]: enclosure {meas:.3f} um < {req:.3f} um {layer} inside nwell between {a} and {b}"
            )
    return "\n".join(lines)


def parse_drc_report(report: str) -> List[DRCViolation]:
    violations: List[DRCViolation] = []
    for raw in report.splitlines():
        raw = raw.strip()
        if not raw:
            continue
        matched = False
        for rid, cat, pat in _RULE_PATTERNS:
            m = pat.search(raw)
            if not m:
                continue
            meas = float(m.group("meas"))
            req = float(m.group("req"))
            if cat == "spacing":
                layer = m.group("layer")
                a, b = m.group("a"), m.group("b")
            elif cat == "width":
                layer = m.group("layer")
                a, b = m.group("a"), m.group("a")
            else:
                layer = m.group("inner")
                a, b = m.group("a"), m.group("b")
            violations.append(
                DRCViolation(
                    rule_id=rid,
                    category=cat,
                    layer=layer,
                    measured_um=meas,
                    required_um=req,
                    shapes=(a, b),
                    margin_um=meas - req,
                )
            )
            matched = True
            break
        if not matched:
            violations.append(
                DRCViolation(
                    rule_id="UNK",
                    category="unknown",
                    layer="unknown",
                    measured_um=float("nan"),
                    required_um=float("nan"),
                    shapes=("?", "?"),
                    margin_um=float("nan"),
                )
            )
    return violations


def suggest_fix(v: DRCViolation) -> str:
    if v.category == "spacing":
        delta = v.required_um - v.measured_um
        return (
            f"Increase separation between {v.shapes[0]} and {v.shapes[1]} on {v.layer} by ≥ {delta:.3f} µm "
            "(move bbox edge or reroute)."
        )
    if v.category == "width":
        delta = v.required_um - v.measured_um
        return f"Widen shape {v.shapes[0]} on {v.layer} by ≥ {delta:.3f} µm (add jog or fat metal)."
    if v.category == "enclosure":
        delta = v.required_um - v.measured_um
        return (
            f"Expand enclosing polygon around ({v.shapes[0]},{v.shapes[1]}) pair on {v.layer} by ≥ {delta:.3f} µm halo."
        )
    return "Manual review: unknown rule family."


def proofread_drc(violations: Sequence[DRCViolation]) -> Dict[str, List[Dict[str, str]]]:
    # Systematic category sweep: deterministic ordering for agent traceability.
    buckets: Dict[str, List[DRCViolation]] = {}
    for v in violations:
        buckets.setdefault(v.category, []).append(v)
    plan: Dict[str, List[Dict[str, str]]] = {}
    for cat in sorted(buckets.keys()):
        items = sorted(buckets[cat], key=lambda x: (x.layer, x.margin_um))
        plan[cat] = [
            {
                "rule": v.rule_id,
                "layer": v.layer,
                "margin_um": f"{v.margin_um:.4f}",
                "fix": suggest_fix(v),
            }
            for v in items
        ]
    return plan


report = synthesize_mock_drc_report(14, seed=3)
violations = parse_drc_report(report)
plan = proofread_drc(violations)

print("--- Sample synthetic DRC report (excerpt) ---")
print("\n".join(report.splitlines()[:5]), "\n...")
print("\n--- Parsed violations:", len(violations))
for v in violations[:3]:
    print(v.category, v.layer, "margin", f"{v.margin_um:.3f}", "→", suggest_fix(v)[:90] + "...")

print("\n--- Proofreading sweep (category keys) ---")
for cat, rows in plan.items():
    print(f"{cat}: {len(rows)} items")


## 10D.5 Placement optimization — simulated annealing with analog cost

We co-optimize an **abstract block placement** of device centroids $\{(x_i,y_i)\}_{i=1}^n$:

$$
\mathcal{C}(\mathbf{x}) = \alpha\, \mathrm{HPWL}(\mathbf{x}) + \beta\, \mathcal{S}_{\mathrm{sym}}(\mathbf{x}) + \gamma\, \mathrm{Area}(\mathbf{x}) + \delta\, \mathcal{P}_{\mathrm{mirror}}(\mathbf{x}),
$$

- **HPWL:** sum over nets of half-perimeter of bounding box of connected device centers (proxy for routing in absence of a detailed router).
- **Symmetry penalty:** for each inferred pair $(i,j)$, penalize deviation from vertical axis $x_0$ and **y-misalignment**.
- **Area:** area of axis-aligned bounding box of all devices (padding included).
- **Mirror proximity $\mathcal{P}_{\mathrm{mirror}}$:** for each **current-mirror cluster** (shared gate net), penalize variance about the cluster centroid (encourages tight placement).

**SA moves:** Gaussian perturbation of one device at a time with reflecting bounds. Temperature **geometrically cooled**.

The next cell runs the optimizer and **plots** (i) cost trajectory, (ii) selected placement snapshots.

---


In [ ]:
# --- Simulated annealing placement (abstract) + dark-theme visuals ---

@dataclass
class PlacementProblem:
    names: List[str]
    nets: Dict[str, List[int]]  # net -> device indices
    sym_pairs: List[Tuple[int, int]]
    mirror_groups: List[List[int]]
    bounds: Tuple[float, float, float, float]  # xmin, xmax, ymin, ymax


def hpwl(pos: np.ndarray, nets: Dict[str, List[int]]) -> float:
    wl = 0.0
    for _, idxs in nets.items():
        pts = pos[idxs, :]
        wl += float(np.max(pts[:, 0]) - np.min(pts[:, 0]) + np.max(pts[:, 1]) - np.min(pts[:, 1]))
    return wl


def sym_penalty(pos: np.ndarray, pairs: Sequence[Tuple[int, int]], x0: float) -> float:
    pen = 0.0
    for i, j in pairs:
        xi, yi = pos[i]
        xj, yj = pos[j]
        pen += ((xi + xj) / 2.0 - x0) ** 2 + (yi - yj) ** 2
    return float(pen)


def bbox_area(pos: np.ndarray, pad: float = 0.2) -> float:
    xmin, xmax = float(np.min(pos[:, 0])), float(np.max(pos[:, 0]))
    ymin, ymax = float(np.min(pos[:, 1])), float(np.max(pos[:, 1]))
    w = (xmax - xmin) + 2 * pad
    h = (ymax - ymin) + 2 * pad
    return w * h


def mirror_proximity_penalty(pos: np.ndarray, groups: Sequence[List[int]]) -> float:
    # Tighten clusters that share gate nets (current mirrors): squared distance to centroid.
    pen = 0.0
    for grp in groups:
        pts = pos[np.array(grp, dtype=int), :]
        c = np.mean(pts, axis=0)
        pen += float(np.sum((pts - c) ** 2))
    return pen


def total_cost(
    pos: np.ndarray,
    prob: PlacementProblem,
    alpha: float,
    beta: float,
    gamma: float,
    delta: float,
    x0: float,
) -> float:
    return (
        alpha * hpwl(pos, prob.nets)
        + beta * sym_penalty(pos, prob.sym_pairs, x0)
        + gamma * bbox_area(pos)
        + delta * mirror_proximity_penalty(pos, prob.mirror_groups)
    )


def random_placement(prob: PlacementProblem, rng: np.random.Generator) -> np.ndarray:
    xmin, xmax, ymin, ymax = prob.bounds
    n = len(prob.names)
    return np.column_stack([rng.uniform(xmin, xmax, size=n), rng.uniform(ymin, ymax, size=n)])


def sa_place(
    prob: PlacementProblem,
    iters: int = 6000,
    seed: int = 42,
    alpha: float = 1.0,
    beta: float = 2.5,
    gamma: float = 0.08,
    delta: float = 0.35,
) -> Tuple[np.ndarray, np.ndarray, List[np.ndarray]]:
    rng = np.random.default_rng(seed)
    xmin, xmax, ymin, ymax = prob.bounds
    x0 = 0.5 * (xmin + xmax)

    pos = random_placement(prob, rng)
    cur = total_cost(pos, prob, alpha, beta, gamma, delta, x0)
    best_pos = pos.copy()
    best = cur

    T0 = 5.0
    T = T0
    cool = 0.9992

    costs = np.zeros(iters)
    snap_idx = {0, iters // 4, iters // 2, (3 * iters) // 4, iters - 1}
    snaps: Dict[int, np.ndarray] = {}

    sigma0 = 0.35
    for t in range(iters):
        costs[t] = cur
        if t in snap_idx:
            snaps[t] = pos.copy()

        i = int(rng.integers(0, len(prob.names)))
        prop = pos.copy()
        sigma = sigma0 * (T / T0) + 0.05
        prop[i, 0] += float(rng.normal(0, sigma))
        prop[i, 1] += float(rng.normal(0, sigma))
        prop[i, 0] = float(np.clip(prop[i, 0], xmin, xmax))
        prop[i, 1] = float(np.clip(prop[i, 1], ymin, ymax))

        new_c = total_cost(prop, prob, alpha, beta, gamma, delta, x0)
        d = new_c - cur
        if d < 0 or rng.random() < math.exp(-d / max(1e-9, T)):
            pos = prop
            cur = new_c
            if cur < best:
                best = cur
                best_pos = pos.copy()
        T *= cool

    ordered_snaps = [snaps[k] for k in sorted(snaps)]
    return best_pos, costs, ordered_snaps


def build_problem_from_constraints(netlist: str) -> PlacementProblem:
    devs = parse_devices(netlist)
    names = sorted(devs.keys())
    idx = {nm: k for k, nm in enumerate(names)}

    nets: Dict[str, List[int]] = {}
    for nm, dv in devs.items():
        for pin in dv.nodes:
            nets.setdefault(pin, []).append(idx[nm])

    cs = extract_constraints(netlist)
    sym_pairs_idx: List[Tuple[int, int]] = []
    for a, b in cs.symmetry_pairs:
        if a in idx and b in idx:
            sym_pairs_idx.append((idx[a], idx[b]))

    mirror_idx: List[List[int]] = []
    for grp in cs.mirror_groups:
        g2 = [idx[x] for x in grp if x in idx]
        if len(g2) >= 2:
            mirror_idx.append(g2)

    return PlacementProblem(
        names=names,
        nets={k: sorted(set(v)) for k, v in nets.items()},
        sym_pairs=sym_pairs_idx,
        mirror_groups=mirror_idx,
        bounds=(0.0, 8.0, 0.0, 6.0),
    )


prob = build_problem_from_constraints(EXAMPLE_NETLIST)
best_pos, cost_trace, snapshots = sa_place(prob, iters=7000, seed=9, alpha=1.0, beta=3.0, gamma=0.06)

fig = plt.figure(figsize=(13.8, 4.35), facecolor=DARK_BG)
ax0 = fig.add_subplot(1, 6, 1)
ax0.set_facecolor(DARK_BG)
ax0.plot(cost_trace, color=ACCENT, lw=1.2)
ax0.set_title("SA cost trajectory")
ax0.set_xlabel("iteration")
ax0.set_ylabel("cost")
ax0.grid(True)

titles = ["t=0", "t≈25%", "t≈50%", "t≈75%", "t=end"]
xmin_b, xmax_b, ymin_b, ymax_b = prob.bounds
x0_axis = 0.5 * (xmin_b + xmax_b)
for k, (P, title) in enumerate(zip(snapshots, titles), start=2):
    ax = fig.add_subplot(1, 6, k)
    ax.set_facecolor(DARK_BG)
    for spine in ax.spines.values():
        spine.set_color("#30363d")
    ax.tick_params(colors="#8b949e", labelsize=8)
    ax.scatter(P[:, 0], P[:, 1], c=CYAN, s=48, edgecolors="#30363d")
    for idx, nm in enumerate(prob.names):
        ax.text(P[idx, 0] + 0.08, P[idx, 1] + 0.08, nm, color="#c9d1d9", fontsize=7)
    ax.axvline(x0_axis, color=AMBER, ls="--", lw=1.0, alpha=0.85)
    ax.set_title(title, color="#c9d1d9", fontsize=9)
    ax.set_xlim(xmin_b, xmax_b)
    ax.set_ylim(ymin_b, ymax_b)
    ax.set_aspect("equal", adjustable="box")

fig.suptitle("Placement optimization — convergence + geometry snapshots", color="#c9d1d9", y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

window = 200
roll_min = np.minimum.accumulate(cost_trace)
smooth = np.convolve(cost_trace, np.ones(window) / window, mode="valid")
fig2 = go.Figure()
fig2.add_trace(go.Scatter(y=cost_trace, name="current cost", line=dict(color=ACCENT, width=1)))
fig2.add_trace(go.Scatter(y=roll_min, name="best-so-far", line=dict(color=GREEN, width=2)))
fig2.add_trace(
    go.Scatter(
        y=smooth,
        x=list(range(window - 1, len(smooth) + window - 1)),
        name=f"{window}-iter MA",
        line=dict(color=PURPLE, width=2),
    )
)
fig2.update_layout(
    title="Placement quality — noisy acceptance vs monotonic best",
    xaxis_title="iteration",
    yaxis_title="cost",
    template="plotly_dark",
    paper_bgcolor=DARK_BG,
    plot_bgcolor="#161b22",
)
fig2.show()


## 10D.6 End-to-end layout flow — pipeline and **sign-off**

**Stages (conceptual):**

```mermaid
flowchart LR
  N[Verified netlist G] --> C[Constraint extraction]
  C --> P[Block placement SA / analytical]
  P --> R[Abstract + detailed routing]
  R --> V[DRC / LVS / ERC]
  V -->|violations| F[Fix suggestions]
  F --> P
  V -->|clean| S[Sign-off + extraction]
```

**Agent roles:**

- **Constraint agent:** maintains `LayoutConstraintSet` JSON as *single source of truth*.
- **Placer/Router agents:** emit geometry edits or script deltas; never bypass DRC.
- **Verification agent:** runs proofreading policy, schedules incremental reruns.

The next cell draws a **Sankey** view of *mass flow* through stages (toy throughput numbers for pedagogy).

---


In [ ]:
# --- Full pipeline visualization (Plotly Sankey, dark theme) ---

labels = [
    "Netlist G",
    "Constraints",
    "Placement",
    "Routing",
    "DRC/LVS",
    "Sign-off",
    "Repair loop",
]
G, C, P, R, V, S, L = range(7)

sources = [G, C, P, R, V, V]
targets = [C, P, R, V, S, L]
values = [100, 100, 100, 100, 72, 28]
colors_link = [
    "rgba(88,166,255,0.45)",
    "rgba(121,192,255,0.45)",
    "rgba(63,185,80,0.45)",
    "rgba(188,140,255,0.45)",
    "rgba(63,185,80,0.55)",
    "rgba(248,81,73,0.55)",
]

fig = go.Figure(
    data=[
        go.Sankey(
            arrangement="snap",
            node=dict(
                pad=18,
                thickness=22,
                line=dict(color="#30363d", width=1),
                label=labels,
                color=["#58a6ff", "#79c0ff", "#3fb950", "#bc8cff", "#d29922", "#3fb950", "#f85149"],
            ),
            link=dict(source=sources, target=targets, value=values, color=colors_link),
        )
    ]
)
fig.update_layout(
    title="End-to-end layout flow — illustrative throughput (toy)",
    font=dict(color="#c9d1d9"),
    paper_bgcolor=DARK_BG,
    plot_bgcolor=DARK_BG,
)
fig.show()

print("Pipeline stages: Netlist → Constraints → Placement → Routing → DRC/LVS → (Sign-off | Repair)")


## 10D.7 LVS proofreading (mock) — from **text report** to **classified mismatches**

LVS tools emit **DEVICE**, **NET**, and **PROPERTY** sections. A robust agent **first classifies** lines, then **grounds** references to layout/schematic IDs (not done geometrically here).

We demonstrate **parsing** and a **remediation hint** table analogous to DRC.

---


In [ ]:
# --- Mock LVS report + structured remediation hints ---

@dataclass
class LVSIssue:
    kind: str
    detail: str


def parse_lvs_report(text: str) -> List[LVSIssue]:
    issues: List[LVSIssue] = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        u = line.upper()
        if u.startswith("DEVICE MISMATCH"):
            issues.append(LVSIssue("device", line))
        elif u.startswith("NET MISMATCH"):
            issues.append(LVSIssue("net", line))
        elif u.startswith("PIN"):
            issues.append(LVSIssue("pin", line))
    return issues


LVS_TEXT = textwrap.dedent('''
    DEVICE MISMATCH: schematic nmos XMN2 != layout nmos M2 (finger count)
    NET MISMATCH: net N_OUTP has extra tap connection in layout
    PIN WARNING: subcell ota_in pin VDD unrouted to top metal
''').strip()

hints = {
    "device": "Revisit series/parallel folding and parameter propagation; align `nf`/`m` with layout stripes.",
    "net": "Search for unintended shorts/opens in guard-ring ties or dummy diffusions; compare extracted DSPF.",
    "pin": "Promote label to met2/met3; ensure text-specific LVS map includes pin name.",
}

issues = parse_lvs_report(LVS_TEXT)
print("--- Parsed LVS issues ---")
for iss in issues:
    print(f"[{iss.kind}] {iss.detail}\n  → {hints.get(iss.kind, 'n/a')}\n")


## 10D.8 Takeaways — what a **production** agent stack adds

- **Real ALIGN/OpenROAD runs** replace mocks with **artifact capture** (GDS, DEF, LEF, logs) and **hash-locked** reproducibility.
- **Incremental verification** uses **tile bounding boxes** from violation coordinates—our mock omits geometry but your parser should retain `(x,y)` when available.
- **Constraint DSL:** symmetry groups, **matching rank**, **shield nets**, and **current-gradient bounds** should live in versioned JSON **next to** the netlist.
- **Human-in-the-loop:** final sign-off remains **designer-owned**; agents propose, tools prove.

### Suggested reading

- ALIGN project documentation (constraint-driven analog layout generation).
- OpenROAD flow scripts and OpenDB schema for mixed-signal integration.
- Classic matching literature: McNeill / Hastings on mismatch, Kinget on analog layout symmetry.

---


In [ ]:
# --- Self-check: deterministic engines ---
assert len(violations) >= 10
assert "enclosure" in plan and "spacing" in plan
assert best_pos.shape == (len(prob.names), 2)
print("Deterministic checks passed (violations, DRC categories, placement shape).")
